# 🦀 Day 3: Process Management — std::process::Command
---

Today we explore **running external programs** from Rust.

- **Command**: `std::process::Command` for spawning child processes
- **Output**: Capture stdout, stderr via `Output` struct
- **Piping**: Connect stdout of one command to stdin of another
- **Environment**: Set env vars for child processes
- **Status vs output**: `status()` waits for exit; `output()` captures streams

In [ ]:
use std::process::Command;

// Windows: use cmd /C for shell commands
#[cfg(windows)]
fn run_echo() -> std::io::Result<std::process::Output> {
    Command::new("cmd").args(["/C", "echo", "hello from Rust"]).output()
}

#[cfg(not(windows))]
fn run_echo() -> std::io::Result<std::process::Output> {
    Command::new("echo").arg("hello from Rust").output()
}

let output = run_echo().expect("Failed to run");
println!("Status: {:?}", output.status);
println!("stdout: {}", String::from_utf8_lossy(&output.stdout));
println!("stderr: {}", String::from_utf8_lossy(&output.stderr));

In [ ]:
// status() vs output()
// status() — just wait for exit code, don't capture streams
// output() — wait and capture stdout + stderr

#[cfg(windows)]
let status = Command::new("cmd").args(["/C", "exit", "0"]).status().unwrap();
#[cfg(not(windows))]
let status = Command::new("true").status().unwrap();

println!("Exit success: {}", status.success());
println!("Exit code: {:?}", status.code());

In [ ]:
// Environment variables for child process
let output = Command::new("cmd")
    .args(["/C", "echo", "%MY_VAR%"])
    .env("MY_VAR", "Hello from env")
    .output()
    .expect("Failed");

// On Unix you'd use: Command::new("sh").args(["-c", "echo $MY_VAR"])
println!("Output: {}", String::from_utf8_lossy(&output.stdout).trim());

In [ ]:
// Piping: stdout of cmd1 -> stdin of cmd2
// Command::new("cmd1").stdout(Stdio::piped())
//     .spawn()?
//     .stdout
//     .take()
// -> pass to Command::new("cmd2").stdin(...)

use std::process::{Command, Stdio};

#[cfg(windows)]
fn pipe_example() {
    let echo = Command::new("cmd")
        .args(["/C", "echo", "hello pipe"])
        .stdout(Stdio::piped())
        .spawn()
        .unwrap();
    let out = Command::new("cmd")
        .args(["/C", "findstr", "hello"])
        .stdin(echo.stdout.unwrap())
        .output()
        .unwrap();
    println!("{}", String::from_utf8_lossy(&out.stdout));
}

#[cfg(not(windows))]
fn pipe_example() {
    let echo = Command::new("echo").arg("hello pipe").stdout(Stdio::piped()).spawn().unwrap();
    let out = Command::new("grep").arg("hello").stdin(echo.stdout.unwrap()).output().unwrap();
    println!("{}", String::from_utf8_lossy(&out.stdout));
}

pipe_example();

In [ ]:
// Simple task runner: run a sequence of commands
let commands: Vec<&str> = vec!["echo step 1", "echo step 2", "echo step 3"];

for (i, cmd) in commands.iter().enumerate() {
    println!("Running: {}", cmd);
    #[cfg(windows)]
    let status = Command::new("cmd").args(["/C", cmd]).status();
    #[cfg(not(windows))]
    let status = Command::new("sh").args(["-c", cmd]).status();
    match status {
        Ok(s) if s.success() => println!("  ✓ Step {} ok", i + 1),
        Ok(s) => println!("  ✗ Step {} failed: {:?}", i + 1, s.code()),
        Err(e) => println!("  ✗ Error: {}", e),
    }
}

## 📝 Exercise: Task runner from config

Build a task runner that reads commands from a config (e.g. a Vec of strings or a simple file).
For each command: run it, check status, and stop on first failure (or continue based on a flag).

In [ ]:
// YOUR CODE HERE
// let config = vec!["echo a", "echo b", "echo c"];
// for cmd in config {
//     let status = Command::new("cmd").args(["/C", cmd]).status()?;
//     if !status.success() { return Err(...); }
// }
// Ok(())

## 🎯 Key Takeaways

1. **Command::new(program).arg(x).output()** runs a process and captures output
2. On Windows use `cmd /C` for shell commands; on Unix use `sh -c`
3. **Output** struct has `status`, `stdout`, `stderr` (Vec<u8>)
4. **status()** waits for exit only; **output()** captures streams
5. **Stdio::piped()** enables piping between processes
6. **env()** sets environment variables for the child

---
**Tomorrow:** File system operations with walkdir